# Brouillons et Expérimentations ML
Ce notebook contient mes premiers essais avec le dataset de recettes. J'ai testé plusieurs approches (CountVectorizer vs TF-IDF) et essayé de faire du clustering (K-Means) avant de me rendre compte que la similarité cosinus avec TF-IDF était la plus pertinente.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.cluster import KMeans
import re

In [2]:
# Load data
df = pd.read_csv('../backend/recipes.csv')
print(df.shape)
df.head(2)

In [3]:
# Test nettoyage basique
def clean_test(text):
    if pd.isna(text): return ''
    text = re.sub(r'\d+', '', str(text))
    return text.lower()

df['ingredients_clean_v1'] = df['ingredients'].apply(clean_test)

### Test 1: CountVectorizer

In [4]:
cv = CountVectorizer(max_features=500, stop_words='english')
cv_matrix = cv.fit_transform(df['ingredients_clean_v1'])
cv_matrix.shape

In [5]:
# Les résultats avec CountVectorizer donnent trop de poids aux mots fréquents comme 'salt', 'pepper', 'sugar'.
# TF-IDF serait mieux pour pénaliser ces mots communs.

### Test 2: Clustering (K-Means) pour catégoriser les recettes

In [6]:
tfidf = TfidfVectorizer(max_features=1000, stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['ingredients_clean_v1'])

# Essayer de trouver le bon nombre de clusters (K)
inertia = []
K = range(2, 10)
for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(tfidf_matrix)
    inertia.append(kmeans.inertia_)

In [7]:
plt.plot(K, inertia, 'bx-')
plt.xlabel('k')
plt.ylabel('Inertia')
plt.title('Elbow Method For Optimal k')
plt.show()

In [8]:
# Problème : La courbe ne montre pas de "coude" évident (Elbow). Les recettes sont trop variées.
# Conclusion : Au lieu de faire du clustering pour assigner une classe, je vais plutôt faire 
# un système de recommandation direct via Cosine Similarity entre la requête utilisateur et le dataset.